In [13]:
from __future__ import annotations

from pathlib import Path

from ase.io import read
import numpy as np
import plotly.graph_objects as go


def _read_extxyz_frames(path: Path) -> list[dict]:
    """Read the extxyz fields needed for EDA with ASE."""
    atoms_list = read(path, index=':', format='extxyz')
    if not isinstance(atoms_list, list):
        atoms_list = [atoms_list]
    return [
        {
            'symbols': atoms.get_chemical_symbols(),
            'positions': np.asarray(atoms.positions, dtype=float),
            'forces': np.asarray(atoms.get_forces(), dtype=float),
            'energy': float(atoms.get_potential_energy()),
            'dihedrals': atoms.info.get('dihedrals'),
        }
        for atoms in atoms_list
    ]


def show_random_frame(
    path: str | Path, *, seed: int | None = None, force_scale: float = 0.35
):
    """Show one random frame with atom and force hover details."""
    frames = _read_extxyz_frames(Path(path))
    if not frames:
        raise ValueError(f'no frames found in {path}')
    rng = np.random.default_rng(seed)
    frame_index = int(rng.integers(len(frames)))
    frame = frames[frame_index]
    positions = frame['positions']
    forces = frame['forces']
    force_norms = np.linalg.norm(forces, axis=1)
    element_colors = {'C': '#6c5ce7', 'H': '#00bfa6', 'N': '#2f80ed', 'O': '#f2994a'}
    marker_colors = [element_colors.get(symbol, '#999999') for symbol in frame['symbols']]
    customdata = np.column_stack([
        np.arange(len(frame['symbols'])),
        np.asarray(frame['symbols']),
        forces,
        force_norms,
    ])
    atom_trace = go.Scatter3d(
        x=positions[:, 0], y=positions[:, 1], z=positions[:, 2],
        mode='markers',
        marker={'size': 7, 'color': marker_colors, 'line': {'width': 1, 'color': '#222222'}},
        customdata=customdata,
        hovertemplate=(
            'atom %{customdata[0]} (%{customdata[1]})<br>'
            'position: (%{x:.3f}, %{y:.3f}, %{z:.3f}) Å<br>'
            'force: (%{customdata[2]:.3f}, %{customdata[3]:.3f}, %{customdata[4]:.3f}) eV/Å<br>'
            '|force|: %{customdata[5]:.3f} eV/Å<extra></extra>'
        ),
        name='atoms',
    )
    line_x: list[float | None] = []
    line_y: list[float | None] = []
    line_z: list[float | None] = []
    for position, force in zip(positions, forces):
        line_x.extend([position[0], position[0] + force_scale * force[0], None])
        line_y.extend([position[1], position[1] + force_scale * force[1], None])
        line_z.extend([position[2], position[2] + force_scale * force[2], None])
    force_trace = go.Scatter3d(
        x=line_x, y=line_y, z=line_z, mode='lines',
        line={'color': 'crimson', 'width': 4},
        hoverinfo='skip', name='forces (scaled for display)',
    )
    title = f"frame {frame_index} | energy={frame['energy']:.6f} eV"
    if frame['dihedrals'] is not None:
        title += f" | dihedrals={frame['dihedrals']}"
    figure = go.Figure([atom_trace, force_trace])
    for element, color in element_colors.items():
        figure.add_trace(go.Scatter3d(
            x=[None], y=[None], z=[None], mode='markers',
            marker={'size': 7, 'color': color}, name=element,
            hoverinfo='skip', showlegend=True,
        ))
    figure.update_layout(title=title, scene_aspectmode='data')
    figure.show()
    return frame_index, figure


In [14]:
DATA_PATH = Path.cwd() / 'data/dataset_3BPA/train_300K.xyz'
if not DATA_PATH.exists():
    DATA_PATH = Path.cwd().parent / "data/dataset_3BPA/train_300K.xyz"
if not DATA_PATH.exists():
    DATA_PATH = Path.cwd() / 'papers/15-MACE/data/dataset_3BPA/train_300K.xyz'
frame_index, figure = show_random_frame(DATA_PATH, seed=42)
